## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

# Plotting style
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
sns.set_style("whitegrid")

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../figures",        exist_ok=True)

print("Libraries loaded.")

## 2. Load Raw ChEMBL Data

In [ ]:
df = pd.read_csv("../data/raw/newdata_nlrp3.csv", sep=";")

print(f"Raw dataset  : {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(2)

In [ ]:
# Overview of available columns
df.info()

In [ ]:
# Distribution of assay measurement types
print("Standard Type value counts:")
print(df["Standard Type"].value_counts().head(10).to_string())

## 3. Assay Harmonisation

In [ ]:
# ── 3a. Keep human-only NLRP3 assays ────────────────────────────
before = len(df)
df = df[
    (df["Target Organism"] == "Homo sapiens") &
    (df["Assay Organism"]  == "Homo sapiens")
].copy()
print(f"Human-only assays : {before:,} → {len(df):,} rows")

# ── 3b. IC50 only ───────────────────────────────────────────────
before = len(df)
df = df[df["Standard Type"] == "IC50"].copy()
print(f"IC50 only         : {before:,} → {len(df):,} rows")

# ── 3c. nM units only ───────────────────────────────────────────
before = len(df)
df = df[df["Standard Units"] == "nM"].copy()
print(f"nM units only     : {before:,} → {len(df):,} rows")

## 4. Select Relevant Columns

In [ ]:
cols_keep = [
    "Molecule ChEMBL ID",
    "Molecule Name",
    "Molecule Max Phase",
    "Smiles",
    "Molecular Weight",
    "AlogP",
    "#RO5 Violations",
    "Standard Value",
    "pChEMBL Value",
    "Action Type",
    "Document Year",
]

df = df[[c for c in cols_keep if c in df.columns]].copy()
print(f"Columns retained  : {df.shape[1]}")
df.head(2)

## 5. Clean Bioactivity Values

In [ ]:
# Drop rows missing core features
before = len(df)
df = df.dropna(subset=["Smiles", "Standard Value", "Molecular Weight", "AlogP"])
print(f"After dropping NaN : {before:,} → {len(df):,} rows")

# Coerce IC50 to numeric and require positive values
df["Standard Value"] = pd.to_numeric(df["Standard Value"], errors="coerce")
before = len(df)
df = df[df["Standard Value"] > 0].copy()
print(f"Positive IC50 only : {before:,} → {len(df):,} rows")

## 6. Compute pIC50

In [ ]:
# pIC50 = -log10(IC50_M) = -log10(IC50_nM * 1e-9)
df["pIC50"] = -np.log10(df["Standard Value"] * 1e-9)

# Cap extreme outliers (IC50 > 100 µM is practically inactive)
df = df[df["pIC50"] >= 3.0].copy()  # IC50 <= 1 mM

print(f"pIC50 range : {df['pIC50'].min():.2f} – {df['pIC50'].max():.2f}")
print(f"pIC50 mean  : {df['pIC50'].mean():.2f}")
print(f"pIC50 std   : {df['pIC50'].std():.2f}")

## 7. Deduplicate by SMILES (Keep Most Potent)

In [ ]:
before = len(df)
df = (
    df
    .sort_values("pIC50", ascending=False)
    .drop_duplicates(subset="Smiles")
)
print(f"After deduplication (unique SMILES) : {before:,} → {len(df):,} molecules")

## 8. Assign Activity Binary Label

In [ ]:
# Threshold: pIC50 >= 6.0  ↔  IC50 <= 1 µM = Active NLRP3 inhibitor
df["Activity_Label"] = df["pIC50"].apply(
    lambda x: "Active" if x >= 6.0 else "Inactive"
)

print("Activity distribution:")
print(df["Activity_Label"].value_counts().to_string())

## 9. Visualise the Dataset

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# pIC50 distribution
axes[0].hist(df["pIC50"], bins=30, color="steelblue", edgecolor="white")
axes[0].axvline(6.0, color="red", linestyle="--", label="Active threshold (pIC50=6)")
axes[0].set_xlabel("pIC50")
axes[0].set_ylabel("Count")
axes[0].set_title("pIC50 Distribution")
axes[0].legend(fontsize=9)

# Molecular Weight distribution
axes[1].hist(df["Molecular Weight"].astype(float), bins=30, color="coral", edgecolor="white")
axes[1].axvline(500, color="green", linestyle="--", label="Lipinski MW ≤ 500")
axes[1].set_xlabel("Molecular Weight (Da)")
axes[1].set_title("Molecular Weight Distribution")
axes[1].legend(fontsize=9)

# LogP distribution
axes[2].hist(df["AlogP"].astype(float), bins=30, color="mediumseagreen", edgecolor="white")
axes[2].axvline(5, color="purple", linestyle="--", label="Lipinski LogP ≤ 5")
axes[2].set_xlabel("AlogP")
axes[2].set_title("LogP Distribution")
axes[2].legend(fontsize=9)

plt.suptitle("NLRP3 Dataset – Physicochemical Properties", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("../figures/01_dataset_distributions.png", bbox_inches="tight")
plt.show()
print("Figure saved.")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = df["Activity_Label"].value_counts()
ax.bar(counts.index, counts.values, color=["#2ecc71", "#e74c3c"])
ax.set_title("Active vs Inactive NLRP3 Modulators")
ax.set_ylabel("Count")
for i, v in enumerate(counts.values):
    ax.text(i, v + 2, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/02_activity_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# Correlation heatmap of numerical features
num_cols = ["Molecular Weight", "AlogP", "#RO5 Violations", "pIC50"]
corr_df = df[num_cols].apply(pd.to_numeric, errors="coerce")

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_df.corr(), annot=True, fmt=".2f", cmap="coolwarm",
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("../figures/03_correlation_heatmap.png", bbox_inches="tight")
plt.show()

## 10. Save ML-Ready Datasets

In [ ]:
# ── Dataset A: Full features (with pChEMBL, Action Type) ────────
full_cols = ["Smiles", "Molecular Weight", "AlogP", "#RO5 Violations", "pIC50", "Activity_Label"]
if "pChEMBL Value" in df.columns:
    full_cols.insert(4, "pChEMBL Value")
if "Action Type" in df.columns:
    full_cols.insert(-1, "Action Type")

dataset_a = df[[c for c in full_cols if c in df.columns]].copy()
dataset_a.to_csv("../data/processed/nlrp3_ml_ready_dataset.csv", index=False)
print(f"Dataset A (full)         : {dataset_a.shape}  → nlrp3_ml_ready_dataset.csv")

# ── Dataset B: Leakage-free (only raw physico-chemical features) ─
leak_free_cols = ["Smiles", "Molecular Weight", "AlogP", "#RO5 Violations", "pIC50"]
dataset_b = df[leak_free_cols].copy()
dataset_b.to_csv("../data/processed/nlrp3_ml_ready_no_leakage.csv", index=False)
print(f"Dataset B (no-leakage)   : {dataset_b.shape}  → nlrp3_ml_ready_no_leakage.csv")

## 11. Lipinski's Rule of Five Analysis

In [ ]:
df["Molecular Weight"] = pd.to_numeric(df["Molecular Weight"], errors="coerce")
df["AlogP"]            = pd.to_numeric(df["AlogP"],            errors="coerce")
df["#RO5 Violations"]  = pd.to_numeric(df["#RO5 Violations"],  errors="coerce")

ro5_pass = df[df["#RO5 Violations"] == 0]

print("=" * 50)
print("LIPINSKI RULE-OF-FIVE SUMMARY")
print("=" * 50)
print(f"Total unique NLRP3 molecules : {len(df):,}")
print(f"  MW ≤ 500 Da               : {(df['Molecular Weight'] <= 500).sum():,}")
print(f"  LogP ≤ 5                  : {(df['AlogP'] <= 5).sum():,}")
print(f"  0 RO5 violations (oral)   : {len(ro5_pass):,}")
print()
print("Active compound RO5 compliance:")
active_df = df[df["Activity_Label"] == "Active"]
print(f"  Active molecules          : {len(active_df):,}")
print(f"  Active + RO5 compliant    : {(active_df['#RO5 Violations'] == 0).sum():,}")

## 12. Summary

In [ ]:
print("=" * 55)
print("  DATA CLEANING SUMMARY")
print("=" * 55)
print(f"  Final dataset (unique SMILES) : {len(df):,}")
print(f"  Active  (pIC50 ≥ 6.0)        : {(df['Activity_Label']=='Active').sum():,}")
print(f"  Inactive                      : {(df['Activity_Label']=='Inactive').sum():,}")
print(f"  pIC50 range                   : {df['pIC50'].min():.2f} – {df['pIC50'].max():.2f}")
print(f"  MW range (Da)                 : {df['Molecular Weight'].min():.1f} – {df['Molecular Weight'].max():.1f}")
print("=" * 55)
print("Outputs:")
print("  data/processed/nlrp3_ml_ready_dataset.csv")
print("  data/processed/nlrp3_ml_ready_no_leakage.csv")